# Precurso 3 — APIs, JSON y primera llamada a Gemini

Esta notebook completa el precurso. El orden es deliberado:

**credenciales → HTTP → JSON → Gemini → async → validación → integración**

Ejecutá las celdas en secuencia. Las llamadas externas están preparadas para mostrar un mensaje controlado si no hay conexión o falta la API key.


## 0. Preparar el kernel y las dependencias

Seleccioná el kernel del entorno virtual `env`. Después ejecutá la celda siguiente una sola vez. `%pip` instala los paquetes dentro del mismo entorno que utiliza la notebook.

Si todavía no aparece ningún kernel en VS Code, activá `env` en la terminal integrada y ejecutá:

```bash
python -m pip install jupyterlab ipykernel
```


### Antes de ejecutar — instalar dentro del kernel correcto

**Qué vas a hacer:** instalar las librerías utilizadas por esta notebook. `%pip` apunta al entorno del kernel activo.  
**Qué observar:** si termina sin error, las dependencias están disponibles. Si VS Code solicita reiniciar el kernel, aceptá y continuá desde la celda siguiente.  
**Solo hace falta ejecutarla una vez por entorno.**


In [1]:
%pip install -q requests python-dotenv google-genai pydantic
print("✓ Dependencias instaladas. Si VS Code lo solicita, reiniciá el kernel.")


Note: you may need to restart the kernel to use updated packages.
✓ Dependencias instaladas. Si VS Code lo solicita, reiniciá el kernel.



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Antes de ejecutar — confirmar el intérprete

**Qué vas a hacer:** verificar que la notebook usa Python 3.11+ y conocer su carpeta de trabajo.  
**Por qué importa:** `.env`, `.gitignore` y los archivos JSON se buscarán o crearán desde esa carpeta.  
**Qué comprobar:** debería ser la raíz de `mi-primer-proyecto`.


In [2]:
import sys
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")
print(f"Intérprete: {sys.executable}")
print(f"Carpeta actual: {Path.cwd()}")

assert sys.version_info >= (3, 11)


Python: 3.12.7
Intérprete: c:\Users\santinoco\curso-ia-developer\precurso\mi-primer-proyecto\env\Scripts\python.exe
Carpeta actual: c:\Users\santinoco\curso-ia-developer\precurso\mi-primer-proyecto


## 1. Credenciales seguras con `.env`

En la carpeta raíz `mi-primer-proyecto` deben existir:

- `.env`: contiene la clave real y **no se sube a Git**.
- `.env.example`: documenta el nombre de la variable sin incluir el secreto.
- `.gitignore`: excluye `.env`, `env/` y archivos temporales.

Creá `.env` manualmente desde el explorador de VS Code y escribí:

```text
GEMINI_API_KEY=pegá_tu_clave_real_aquí
```

No pegues la clave dentro de una celda: las notebooks también pueden terminar versionadas.


### Antes de ejecutar — preparar archivos seguros

**Qué vas a hacer:** crear la plantilla `.env.example` y completar reglas de `.gitignore`.  
**Qué no hace la celda:** no crea ni escribe tu API key real.  
**Qué observar:** al final se imprimen todas las reglas ignoradas, incluida `.env`.


In [3]:
from pathlib import Path

Path(".env.example").write_text("GEMINI_API_KEY=\n", encoding="utf-8")

gitignore = Path(".gitignore")
reglas = {".env", "env/", ".venv/", "__pycache__/", ".ipynb_checkpoints/"}
existentes = set(gitignore.read_text(encoding="utf-8").splitlines()) if gitignore.exists() else set()
gitignore.write_text("\n".join(sorted(existentes | reglas)) + "\n", encoding="utf-8")

print("✓ .env.example actualizado")
print("✓ .gitignore actualizado")
print(gitignore.read_text(encoding="utf-8"))


✓ .env.example actualizado
✓ .gitignore actualizado
.env
.ipynb_checkpoints/
.venv/
__pycache__/
env/



### Antes de ejecutar — cargar la API key sin mostrarla

**Qué vas a hacer:** leer `GEMINI_API_KEY` desde `.env`.  
**Qué observar:** se informa si existe y su longitud, pero nunca su contenido.  
**Si no la encuentra:** confirmá el nombre exacto del archivo, la variable y la carpeta de trabajo.


In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
gemini_key = os.getenv("GEMINI_API_KEY")

if gemini_key:
    print("✓ GEMINI_API_KEY encontrada")
    print(f"Longitud: {len(gemini_key)} caracteres (el valor no se muestra)")
else:
    print("⚠ GEMINI_API_KEY no encontrada. Creá o revisá el archivo .env.")


✓ GEMINI_API_KEY encontrada
Longitud: 53 caracteres (el valor no se muestra)


## 2. HTTP con `requests`

Una petición tiene método, URL, headers y, a veces, un body. Una respuesta tiene status code, headers y contenido.


### Antes de ejecutar — realizar un GET

**Qué vas a hacer:** enviar parámetros a una API de prueba y convertir su respuesta JSON a un diccionario.  
**Qué mirar:** `timeout` evita esperar indefinidamente y `raise_for_status()` convierte códigos de error en excepciones.  
**Resultado esperado:** status 200 y los parámetros enviados.


In [5]:
import requests

try:
    response = requests.get(
        "https://httpbin.org/get",
        params={"curso": "ai-agent-developer", "parte": 3},
        timeout=10,
    )
    response.raise_for_status()
    datos_get = response.json()
    print("Status:", response.status_code)
    print("Parámetros recibidos:", datos_get["args"])
except requests.RequestException as error:
    datos_get = None
    print(f"No se pudo completar el GET: {error}")


Status: 200
Parámetros recibidos: {'curso': 'ai-agent-developer', 'parte': '3'}


### Antes de ejecutar — realizar un POST

**Qué vas a hacer:** enviar un diccionario como body JSON. `requests` se ocupa de serializarlo y agregar el header adecuado.  
**Qué observar:** el servidor de prueba devuelve el mismo objeto dentro de `json`.  
**Diferencia clave:** GET consulta; POST envía datos para procesar.


In [6]:
payload = {"pregunta": "¿Qué es JSON?", "nivel": "inicial"}

try:
    response = requests.post("https://httpbin.org/post", json=payload, timeout=10)
    response.raise_for_status()
    datos_post = response.json()
    print("Status:", response.status_code)
    print("JSON recibido por el servidor:", datos_post["json"])
except requests.RequestException as error:
    datos_post = None
    print(f"No se pudo completar el POST: {error}")


Status: 200
JSON recibido por el servidor: {'nivel': 'inicial', 'pregunta': '¿Qué es JSON?'}


### Códigos frecuentes

| Código | Interpretación inicial |
|---|---|
| 200 | La solicitud funcionó |
| 400 | La solicitud es inválida |
| 401 / 403 | Faltan credenciales o permisos |
| 404 | El recurso no existe |
| 429 | Se excedió una cuota o límite |
| 500+ | Falló el servicio remoto |


## 3. JSON en Python

JSON es texto. `json.loads` lo convierte a Python y `json.dumps` convierte estructuras Python a texto JSON.


### Antes de ejecutar — convertir JSON a Python

**Qué vas a hacer:** transformar texto JSON con `json.loads`.  
**Qué observar:** `true` pasa a `True`, `null` pasa a `None`, los objetos a `dict` y los arrays a `list`.  
**Probá:** agregá una clave `"intentos": 3` y leela desde `datos`.


In [7]:
import json

json_texto = '''
{
  "modelo": "gemini-2.5-flash",
  "temperature": 0.2,
  "mensajes": [{"role": "user", "content": "Hola"}],
  "activo": true,
  "error": null,
  "intentos": 3
}
'''

datos = json.loads(json_texto)
print(type(datos))
print(datos["modelo"])
print(datos["mensajes"][0]["content"])
print(datos["error"] is None)
print(datos["intentos"])


<class 'dict'>
gemini-2.5-flash
Hola
True
3


### Antes de ejecutar — guardar y recuperar JSON

**Qué vas a hacer:** serializar un diccionario, guardarlo en disco y volver a leerlo.  
**Qué observar:** `ensure_ascii=False` conserva caracteres en español y `indent=2` mejora la lectura.  
**Comprobación:** el `assert` confirma que los datos recuperados son iguales a los originales.


In [8]:
configuracion = {
    "model": "gemini-2.5-flash",
    "temperature": 0.2,
    "max_output_tokens": 300,
}

json_generado = json.dumps(configuracion, indent=2, ensure_ascii=False)
print(json_generado)

ruta_config = Path("config_precurso.json")
ruta_config.write_text(json_generado, encoding="utf-8")
config_recuperada = json.loads(ruta_config.read_text(encoding="utf-8"))

assert config_recuperada == configuracion
print(f"✓ Archivo verificado: {ruta_config.resolve()}")


{
  "model": "gemini-2.5-flash",
  "temperature": 0.2,
  "max_output_tokens": 300
}
✓ Archivo verificado: C:\Users\santinoco\curso-ia-developer\precurso\mi-primer-proyecto\config_precurso.json


## 4. Primera llamada a Gemini

Esta celda solo llama al servicio si encuentra la API key. No imprime ni registra el secreto.


### Antes de ejecutar — llamar a Gemini de forma controlada

**Qué vas a hacer:** crear el cliente, elegir modelo y enviar un prompt.  
**Qué observar:** si falta la key, la celda se omite sin romper la notebook; si existe, imprime el texto generado.  
**Seguridad:** la clave se toma de memoria y nunca se concatena con el prompt ni se imprime.


In [14]:
respuesta_gemini = None

if not gemini_key:
    print("⚠ Llamada omitida: falta GEMINI_API_KEY en .env")
else:
    try:
        from google import genai
        from google.genai import types

        client = genai.Client(api_key=gemini_key)
        respuesta_gemini = client.models.generate_content(
            model="gemini-flash-latest",
            contents="Explicá qué es una API en dos oraciones para una persona que empieza a programar.",
            config=types.GenerateContentConfig(
                temperature=0.2,
                max_output_tokens=500,
            ),
        )
        print(respuesta_gemini.text)
    except Exception as error:
        print(f"La llamada a Gemini no pudo completarse: {type(error).__name__}: {error}")


Una API es como un "mensajero" que permite que dos programas diferentes se comuniquen y


### 4.1 Inspeccionar metadatos sin asumir que siempre existen

Los SDKs pueden devolver campos opcionales. `getattr` permite leerlos de forma segura durante esta primera exploración.


### Antes de ejecutar — inspeccionar metadatos opcionales

**Qué vas a hacer:** buscar información de uso sin asumir que todos los campos existen.  
**Qué observar:** `getattr(..., valor_alternativo)` evita que la exploración falle si cambia o falta un campo.  
**Si no hubo llamada real:** la celda informa que todavía no hay respuesta para inspeccionar.


In [15]:
if respuesta_gemini is not None:
    usage = getattr(respuesta_gemini, "usage_metadata", None)
    if usage:
        print("Input tokens:", getattr(usage, "prompt_token_count", "no disponible"))
        print("Output tokens:", getattr(usage, "candidates_token_count", "no disponible"))
        print("Total tokens:", getattr(usage, "total_token_count", "no disponible"))
    else:
        print("La respuesta no informó usage_metadata")
else:
    print("No hay respuesta real para inspeccionar todavía")


Input tokens: 21
Output tokens: 19
Total tokens: 517


## 5. Introducción a `async` / `await`

No necesitás dominar concurrencia todavía. El objetivo es reconocer la sintaxis y entender que una tarea puede esperar sin bloquear a las demás.


### Antes de ejecutar — observar concurrencia

**Qué vas a hacer:** iniciar dos tareas y esperar ambas mediante `asyncio.gather`.  
**Qué observar:** la notebook permite usar `await` directamente. En un archivo `.py` normalmente se utiliza `asyncio.run(...)`.  
**Objetivo:** reconocer la sintaxis; no necesitás dominar concurrencia todavía.


In [16]:
import asyncio


async def tarea(nombre: str, demora: float) -> str:
    await asyncio.sleep(demora)
    return f"{nombre} terminada"


resultados_async = await asyncio.gather(
    tarea("consulta A", 0.15),
    tarea("consulta B", 0.10),
)

print(resultados_async)


['consulta A terminada', 'consulta B terminada']


## 6. Validación con Pydantic

Pydantic verifica tipos y reglas antes de que una aplicación utilice datos externos.


### Antes de ejecutar — validar antes de utilizar datos

**Qué vas a hacer:** definir campos, valores predeterminados y límites con Pydantic.  
**Qué observar:** el primer objeto se acepta; el segundo se rechaza por pregunta corta y temperatura fuera de rango.  
**Idea central:** JSON válido no significa dato correcto para tu aplicación.


In [17]:
from pydantic import BaseModel, Field, ValidationError


class ConsultaIA(BaseModel):
    pregunta: str = Field(min_length=3)
    modelo: str = "gemini-2.5-flash"
    temperature: float = Field(default=0.2, ge=0, le=2)


consulta_valida = ConsultaIA(
    pregunta="¿Qué diferencia hay entre JSON y un diccionario?",
    temperature=0.3,
)
print(consulta_valida.model_dump())

try:
    ConsultaIA(pregunta="x", temperature=8)
except ValidationError as error:
    print("✓ Pydantic rechazó correctamente los datos inválidos")
    print(error.errors())


{'pregunta': '¿Qué diferencia hay entre JSON y un diccionario?', 'modelo': 'gemini-2.5-flash', 'temperature': 0.3}
✓ Pydantic rechazó correctamente los datos inválidos
[{'type': 'string_too_short', 'loc': ('pregunta',), 'msg': 'String should have at least 3 characters', 'input': 'x', 'ctx': {'min_length': 3}, 'url': 'https://errors.pydantic.dev/2.13/v/string_too_short'}, {'type': 'less_than_equal', 'loc': ('temperature',), 'msg': 'Input should be less than or equal to 2', 'input': 8, 'ctx': {'le': 2.0}, 'url': 'https://errors.pydantic.dev/2.13/v/less_than_equal'}]


## 7. Proyecto integrador — Cliente Gemini seguro

El modo `dry_run=True` permite verificar configuración y prompt sin consumir la API. Después podés cambiarlo a `False` para realizar la llamada real.


### Antes de ejecutar — construir un cliente seguro

**Qué vas a construir:** un cliente con resultado tipado, latencia, manejo de errores y modo de prueba.  
**Por qué empieza en `dry_run`:** permite validar el flujo sin consumir cuota ni depender de Internet.  
**Cómo leerlo:** revisá primero `ResultadoIA`, después el constructor y finalmente `consultar`.


In [19]:
from dataclasses import dataclass
from time import perf_counter


@dataclass
class ResultadoIA:
    ok: bool
    texto: str | None
    error: str | None
    latencia_ms: float


class ClienteGemini:
    def __init__(self, api_key: str | None, model: str = "gemini-2.5-flash") -> None:
        self.api_key = api_key
        self.model = model

    def consultar(self, consulta: ConsultaIA, dry_run: bool = False) -> ResultadoIA:
        inicio = perf_counter()

        if dry_run:
            texto = f"DRY RUN | model={consulta.modelo} | pregunta={consulta.pregunta}"
            return ResultadoIA(True, texto, None, (perf_counter() - inicio) * 1000)

        if not self.api_key:
            return ResultadoIA(False, None, "Falta GEMINI_API_KEY", (perf_counter() - inicio) * 1000)

        try:
            from google import genai
            from google.genai import types

            cliente = genai.Client(api_key=self.api_key)
            respuesta = cliente.models.generate_content(
                model=consulta.modelo,
                contents=consulta.pregunta,
                config=types.GenerateContentConfig(temperature=consulta.temperature),
            )
            return ResultadoIA(True, respuesta.text, None, (perf_counter() - inicio) * 1000)
        except Exception as error:
            return ResultadoIA(
                False,
                None,
                f"{type(error).__name__}: {error}",
                (perf_counter() - inicio) * 1000,
            )


cliente_seguro = ClienteGemini(gemini_key)
resultado_dry = cliente_seguro.consultar(consulta_valida, dry_run=True)
print(resultado_dry)


ResultadoIA(ok=True, texto='DRY RUN | model=gemini-2.5-flash | pregunta=¿Qué diferencia hay entre JSON y un diccionario?', error=None, latencia_ms=0.001300009898841381)


### Antes de ejecutar — preparar archivos seguros

**Qué vas a hacer:** crear la plantilla `.env.example` y completar reglas de `.gitignore`.  
**Qué no hace la celda:** no crea ni escribe tu API key real.  
**Qué observar:** al final se imprimen todas las reglas ignoradas, incluida `.env`.


In [20]:
assert resultado_dry.ok is True
assert resultado_dry.texto is not None
assert "DRY RUN" in resultado_dry.texto
assert Path(".env.example").exists()
assert ".env" in Path(".gitignore").read_text(encoding="utf-8")
assert config_recuperada["model"] == "gemini-2.5-flash"

print("✓ NOTEBOOK 3 COMPLETADA EN MODO SEGURO")
print("Para probar la API real, configurá .env y ejecutá la sección 4.")


✓ NOTEBOOK 3 COMPLETADA EN MODO SEGURO
Para probar la API real, configurá .env y ejecutá la sección 4.


## Si algo falla

- **`ModuleNotFoundError`**: ejecutá nuevamente la celda `%pip install ...` y reiniciá el kernel.
- **No encuentra `.env`**: confirmá que VS Code abrió la carpeta raíz `mi-primer-proyecto`.
- **401 / 403**: revisá la API key y sus permisos; no la pegues en la notebook.
- **429**: esperá y reintentá más tarde; puede haberse alcanzado una cuota.
- **Error de conexión**: comprobá Internet y volvé a ejecutar solo esa celda.
- **Variables no definidas**: elegí **Restart Kernel and Run All Cells**.

> La notebook es el entorno guiado de aprendizaje. En el curso principal convertirás las partes estables en módulos `.py`, tests y aplicaciones completas.
